In [1]:
!nvidia-smi

Thu Nov 27 13:53:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:01:00.0 Off |                  Off |
| 30%   28C    P8             21W /  300W |   17658MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import paddle
gpu_available  = paddle.device.is_compiled_with_cuda()
print("GPU available:", gpu_available)

/mnt/tank/scratch/plutskyi/anaconda3/envs/image_pdf/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


GPU available: True


In [3]:
paddle.utils.run_check()

Running verify PaddlePaddle program ... 


/mnt/tank/scratch/plutskyi/anaconda3/envs/image_pdf/lib/python3.12/site-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(
I1127 13:53:14.601660 373751 pir_interpreter.cc:1524] New Executor is Running ...
W1127 13:53:14.602602 373751 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 8.6, Driver API Version: 12.4, Runtime API Version: 12.6
I1127 13:53:14.607138 373751 pir_interpreter.cc:1547] pir interpreter is running by multi-thread mode ...


PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


Делаем аннотацию для всех папок тасков и созраняем отдельным файлом, далее мы будем с ним работать для создания COCO 1.0 json

In [6]:
import json
from pathlib import Path
from PIL import Image
from paddleocr import LayoutDetection

base_dir = Path("/mnt/tank/scratch/plutskyi/cvat_share_files/SEM_TEM_DETECT/")
out_dir = Path("/mnt/tank/scratch/plutskyi/cvat_share_files/TASKS_JSONS_PADDLE/")
out_dir.mkdir(parents=True, exist_ok=True)

model = LayoutDetection(model_name="PP-DocLayout_plus-L")

for task_folder in sorted(base_dir.iterdir()):
    if task_folder.is_dir(): 
        task_name = task_folder.name
        print(f"Processing folder: {task_name}...")

        out_json_path = out_dir / f"layout_{task_name}.json"
        folder_results = []

        images = sorted(task_folder.glob("*.png"))
        if not images:
            continue
            
        for img_path in images:
            output = model.predict(str(img_path), batch_size=1, layout_nms=True)
            for res in output:
                j = res.json 
                if isinstance(j, dict):
                    j['file_name'] = img_path.name
                folder_results.append(j)

        with out_json_path.open("w", encoding="utf-8") as f:
            json.dump(folder_results, f, ensure_ascii=False, indent=2)
        print(f"Saved {len(folder_results)} annotations to {out_json_path.name}")


/mnt/tank/scratch/plutskyi/anaconda3/envs/image_pdf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using official model (PP-DocLayout_plus-L), the model files will be automatically downloaded and saved in `/nfs/home/plutskyi/.paddlex/official_models/PP-DocLayout_plus-L`.
Fetching 6 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:03<00:00,  1.77it/s]


Processing folder: task_1...
Saved 125 annotations to layout_task_1.json
Processing folder: task_10...
Saved 209 annotations to layout_task_10.json
Processing folder: task_11...
Saved 236 annotations to layout_task_11.json
Processing folder: task_12...
Saved 234 annotations to layout_task_12.json
Processing folder: task_13...
Saved 188 annotations to layout_task_13.json
Processing folder: task_14...
Saved 245 annotations to layout_task_14.json
Processing folder: task_15...
Saved 228 annotations to layout_task_15.json
Processing folder: task_16...
Saved 284 annotations to layout_task_16.json
Processing folder: task_17...
Saved 208 annotations to layout_task_17.json
Processing folder: task_18...
Saved 258 annotations to layout_task_18.json
Processing folder: task_19...
Saved 232 annotations to layout_task_19.json
Processing folder: task_2...
Saved 175 annotations to layout_task_2.json
Processing folder: task_20...
Saved 303 annotations to layout_task_20.json
Processing folder: task_21...

In [7]:
folder_results[0]['res']['input_path']

'/mnt/tank/scratch/plutskyi/cvat_share_files/SEM_TEM_DETECT/task_9/155_1-s2.0-S0928098723003111-main_page_0001.png'

In [8]:
for res in folder_results:
    print(res['res']['input_path'].rsplit('/', 1)[-1])

155_1-s2.0-S0928098723003111-main_page_0001.png
155_1-s2.0-S0928098723003111-main_page_0002.png
155_1-s2.0-S0928098723003111-main_page_0003.png
155_1-s2.0-S0928098723003111-main_page_0004.png
155_1-s2.0-S0928098723003111-main_page_0005.png
155_1-s2.0-S0928098723003111-main_page_0006.png
155_1-s2.0-S0928098723003111-main_page_0007.png
155_1-s2.0-S0928098723003111-main_page_0008.png
155_1-s2.0-S0928098723003111-main_page_0009.png
155_1-s2.0-S0928098723003111-main_page_0010.png
155_1-s2.0-S0928098723003111-main_page_0011.png
155_1-s2.0-S0928098723003111-main_page_0012.png
155_1-s2.0-S0928098723003111-main_page_0013.png
155_1-s2.0-S0928098723003111-main_page_0014.png
155_1-s2.0-S0928098723003111-main_page_0015.png
155_motskin2009_page_0001.png
155_motskin2009_page_0002.png
155_motskin2009_page_0003.png
155_motskin2009_page_0004.png
155_motskin2009_page_0005.png
155_motskin2009_page_0006.png
155_motskin2009_page_0007.png
155_motskin2009_page_0008.png
155_motskin2009_page_0009.png
155_motski

ТЕст для одной папки и одного файла

In [10]:
import json
from pathlib import Path
from PIL import Image

img_dir = Path("/mnt/tank/scratch/plutskyi/cvat_share_files/SEM_TEM_DETECT/task_9")
raw_path = Path("/mnt/tank/scratch/plutskyi/cvat_share_files/TASKS_JSONS_PADDLE/layout_task_9.json")
coco_out = Path("/mnt/tank/scratch/plutskyi/layout_coco.json")

with raw_path.open("r", encoding="utf-8") as f:
    all_results = json.load(f)

images = []
annotations = []
categories = []

label_to_cat_id = {}
next_cat_id = 1
next_img_id = 1
next_ann_id = 1

for res in all_results:
    img_path = res['res']["input_path"]
    file_name = img_path.rsplit('/', 1)[-1]

    # читаем размеры
    with Image.open(img_dir / file_name) as img:
        width, height = img.size

    image_id = next_img_id
    next_img_id += 1

    images.append({
        "id": image_id,
        "file_name": file_name,
        "width": width,
        "height": height,
    })

    for box in res["res"]["boxes"]:
        x1, y1, x2, y2 = box["coordinate"]
        w = x2 - x1
        h = y2 - y1

        label = box["label"]

        if label in {"figure_title", "number", "header", "formula", 
                     "paragraph_title", "reference", "reference_content", "doc_title", "abstract", 
                     "footer", "text", "footnote", "table", "formula_number", "aside_text"}:
            continue

        # регистрируем категорию
        if label not in label_to_cat_id:
            label_to_cat_id[label] = next_cat_id
            categories.append({
                "id": next_cat_id,
                "name": label,
                "supercategory": "layout",
            })
            next_cat_id += 1

        cat_id = label_to_cat_id[label]

        annotations.append({
            "id": next_ann_id,
            "image_id": image_id,
            "category_id": cat_id,
            "bbox": [x1, y1, w, h],  
            "area": float(w * h),
            "iscrowd": 0,
            "score": float(box["score"]),
        })
        next_ann_id += 1

coco = {
    "images": images,
    "annotations": annotations,
    "categories": categories,
}

with coco_out.open("w", encoding="utf-8") as f:
    json.dump(coco, f, ensure_ascii=False, indent=2)


Пробегаемся по всем папкам и создаем json COCO для каждой таски в отдельной папке

In [6]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm  

# === НАСТРОЙКИ ПУТЕЙ ===
BASE_DIR = Path("/mnt/tank/scratch/plutskyi")
IMG_ROOT = BASE_DIR / "cvat_share_files/SEM_TEM_DETECT"
JSON_IN_ROOT = BASE_DIR / "cvat_share_files/TASKS_JSONS_PADDLE"
JSON_OUT_ROOT = BASE_DIR / "cvat_share_files/TASKS_COCO_JSON"

JSON_OUT_ROOT.mkdir(parents=True, exist_ok=True)


CATEGORY_MAP = {
    "microscopy": 1,
    "chart": 2
}

CATEGORIES_INFO = [
    {"id": 1, "name": "microscopy", "supercategory": "layout"},
    {"id": 2, "name": "chart", "supercategory": "layout"}
]

# Находим все файлы вида layout_task_*.json
json_files = list(JSON_IN_ROOT.glob("layout_task_*.json"))

print(f"Найдено {len(json_files)} файлов для обработки.")

for raw_path in tqdm(json_files):
    task_folder_name = raw_path.stem.replace("layout_", "")  # "task_{i}"
    
    img_dir = IMG_ROOT / task_folder_name
    coco_out = JSON_OUT_ROOT / f"{task_folder_name}.json"

    # Проверка, существует ли папка с картинками
    if not img_dir.exists():
        print(f"⚠️ Папка с картинками не найдена: {img_dir}, пропускаем...")
        continue

    # 2. Читаем исходный JSON
    with raw_path.open("r", encoding="utf-8") as f:
        all_results = json.load(f)

    images = []
    annotations = []
    
    next_img_id = 1
    next_ann_id = 1

    for res in all_results:
        img_path_str = res['res']["input_path"]
        file_name = img_path_str.rsplit('/', 1)[-1]
        full_img_path = img_dir / file_name

        # Проверка, существует ли файл картинки (чтобы не падать на Image.open)
        if not full_img_path.exists():
            continue

        try:
            with Image.open(full_img_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"Ошибка чтения картинки {file_name}: {e}")
            continue

        image_id = next_img_id
        next_img_id += 1

        images.append({
            "id": image_id,
            "file_name": file_name,
            "width": width,
            "height": height,
        })

        for box in res["res"]["boxes"]:
            label = box["label"]

            # === ЛОГИКА ФИЛЬТРАЦИИ И ПОДМЕНЫ ИМЕН ===
            target_label = None
            
            if label == "image":
                target_label = "microscopy"
            elif label == "chart":
                target_label = "chart"
            else:
                continue  

            cat_id = CATEGORY_MAP[target_label]

            x1, y1, x2, y2 = box["coordinate"]
            w = x2 - x1
            h = y2 - y1

            annotations.append({
                "id": next_ann_id,
                "image_id": image_id,
                "category_id": cat_id,
                "bbox": [x1, y1, w, h],
                "area": float(w * h),
                "iscrowd": 0,
                "score": float(box["score"]) if "score" in box else 1.0,
                "segmentation": []
            })
            next_ann_id += 1

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": CATEGORIES_INFO,
    }

    with coco_out.open("w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=2)

print("Готово! Файлы сохранены в", JSON_OUT_ROOT)


Найдено 47 файлов для обработки.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 47/47 [00:07<00:00,  6.68it/s]

Готово! Файлы сохранены в /mnt/tank/scratch/plutskyi/cvat_share_files/TASKS_COCO_JSON
